# 03 — Run TEMPTED (R)


In [1]:
suppressPackageStartupMessages({
  library(tempted)
  library(nnet)
})

BATCHES_TO_RUN <- character(0)
RANK <- 5

root <- if (dir.exists("data")) "." else ".."
split_folder <- tail(sort(list.dirs(file.path(root, "data", "splits"), recursive = FALSE)), 1)
output_folder <- file.path(root, "data", "tempted", format(Sys.time(), "%Y%m%d_%H%M%S"))
dir.create(output_folder, recursive = TRUE)

batch_folders <- sort(list.dirs(split_folder, recursive = FALSE))
batch_folders <- batch_folders[grepl("^batch_", basename(batch_folders))]
if (length(BATCHES_TO_RUN)) batch_folders <- batch_folders[basename(batch_folders) %in% BATCHES_TO_RUN]

subject_table <- function(x, meta) {
  x <- as.data.frame(x)
  names(x) <- paste0("factor_", seq_len(ncol(x)))
  x$subject_id <- rownames(x)
  merge(x, unique(meta[c("subject_id", "label")]), by = "subject_id")
}
balanced_accuracy <- function(y, p) mean(sapply(unique(y), function(x) mean(p[y == x] == x)))
macro_f1 <- function(y, p) mean(sapply(unique(y), function(x) {
  pr <- sum(p == x & y == x) / max(1, sum(p == x))
  re <- sum(p == x & y == x) / max(1, sum(y == x))
  if (pr + re == 0) 0 else 2 * pr * re / (pr + re)
}))


In [2]:
run_batch <- function(folder) {
  started <- proc.time()[3]
  batch <- basename(folder)

  read_matrix <- function(name) {
    x <- read.csv(gzfile(file.path(folder, name)), check.names = FALSE)
    rownames(x) <- as.character(x$sample_id)
    x$sample_id <- NULL
    as.matrix(x)
  }

  train_clr <- read_matrix("train_tempted_clr.csv.gz")
  test_clr <- read_matrix("test_tempted_clr.csv.gz")
  train_meta <- read.csv(gzfile(file.path(folder, "train_metadata.csv.gz")), stringsAsFactors = FALSE)
  test_meta <- read.csv(gzfile(file.path(folder, "test_metadata.csv.gz")), stringsAsFactors = FALSE)
  train_meta$sample_id <- as.character(train_meta$sample_id)
  test_meta$sample_id <- as.character(test_meta$sample_id)
  train_meta$subject_id <- as.character(train_meta$subject_id)
  test_meta$subject_id <- as.character(test_meta$subject_id)

  train_data <- format_tempted(train_clr[train_meta$sample_id, ], train_meta$time, train_meta$subject_id,
                               threshold = 1, transform = "none")
  center <- svd_centralize(train_data, r = 1)
  model <- tempted(center$datlist, r = RANK)

  features <- rownames(train_data[[1]])[-1]
  test_data <- format_tempted(test_clr[test_meta$sample_id, features, drop = FALSE],
                              test_meta$time, test_meta$subject_id, threshold = 1, transform = "none")

  train_scores <- subject_table(model$A_hat, train_meta)
  test_scores <- subject_table(est_test_subject(test_data, model, center), test_meta)
  factors <- grep("^factor_", names(train_scores), value = TRUE)

  means <- sapply(train_scores[factors], mean)
  scales <- sapply(train_scores[factors], sd)
  scales[!is.finite(scales) | scales == 0] <- 1
  train_scores[factors] <- scale(train_scores[factors], means, scales)
  test_scores[factors] <- scale(test_scores[factors], means, scales)

  classifier <- multinom(label ~ ., train_scores[c("label", factors)], trace = FALSE)
  predicted <- as.character(predict(classifier, test_scores[factors]))
  truth <- as.character(test_scores$label)

  predictions <- data.frame(batch, method = "TEMPTED", subject_id = test_scores$subject_id, truth, predicted)
  metrics <- data.frame(batch, method = "TEMPTED", status = "success",
                        accuracy = mean(predicted == truth),
                        balanced_accuracy = balanced_accuracy(truth, predicted),
                        macro_f1 = macro_f1(truth, predicted),
                        elapsed_seconds = proc.time()[3] - started, error = "")

  batch_output <- file.path(output_folder, batch)
  dir.create(batch_output)
  B <- as.matrix(model$B_hat)
  if (nrow(B) < ncol(B)) B <- t(B)
  feature_ids <- rownames(B)
  if (is.null(feature_ids)) feature_ids <- features[seq_len(nrow(B))]
  loadings <- data.frame(feature_id = feature_ids, B, check.names = FALSE)
  names(loadings)[-1] <- paste0("factor_", seq_len(ncol(B)))

  write.csv(loadings, gzfile(file.path(batch_output, "feature_loadings.csv.gz")), row.names = FALSE)
  write.csv(train_scores, gzfile(file.path(batch_output, "train_subject_scores.csv.gz")), row.names = FALSE)
  write.csv(test_scores, gzfile(file.path(batch_output, "test_subject_scores.csv.gz")), row.names = FALSE)
  write.csv(predictions, gzfile(file.path(batch_output, "predictions.csv.gz")), row.names = FALSE)
  write.csv(metrics, file.path(batch_output, "metrics.csv"), row.names = FALSE)
  saveRDS(list(model = model, centralization = center), file.path(batch_output, "model.rds"))
  list(metrics = metrics, predictions = predictions)
}

results <- lapply(batch_folders, run_batch)
all_metrics <- do.call(rbind, lapply(results, `[[`, "metrics"))
all_predictions <- do.call(rbind, lapply(results, `[[`, "predictions"))
write.csv(all_metrics, file.path(output_folder, "all_metrics.csv"), row.names = FALSE)
write.csv(all_predictions, gzfile(file.path(output_folder, "all_predictions.csv.gz")), row.names = FALSE)
cat("Saved:", output_folder, "\n")
all_metrics


Calculate the 1th Component

Convergence reached at dif=7.71640387370648e-05, iter=10

Calculate the 2th Component

Convergence reached at dif=0.149915705073199, iter=21

Calculate the 3th Component

Convergence reached at dif=0.00010536039560344, iter=21

Calculate the 4th Component

Convergence reached at dif=9.37703423478806e-05, iter=17

Calculate the 5th Component

Convergence reached at dif=8.93526110782882e-05, iter=13

Calculate the 1th Component

Convergence reached at dif=4.1438545217077e-05, iter=7

Calculate the 2th Component

Convergence reached at dif=0.00090253007803658, iter=21

Calculate the 3th Component

Convergence reached at dif=9.3940865029181e-05, iter=18

Calculate the 4th Component

Convergence reached at dif=0.00237718816856866, iter=21

Calculate the 5th Component

Convergence reached at dif=6.21742879906133e-05, iter=13

Calculate the 1th Component

Convergence reached at dif=7.24021220133872e-05, iter=12

Calculate the 2th Component

Convergence reached at 

Saved: ../data/tempted/20260807_221553 


,batch,method,status,accuracy,balanced_accuracy,macro_f1,elapsed_seconds,error
,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
elapsed,batch_001,TEMPTED,success,0.5238095,0.5238095,0.5228462,8.589,
elapsed1,batch_002,TEMPTED,success,0.5396825,0.5396825,0.5285485,5.565,
elapsed2,batch_003,TEMPTED,success,0.6666667,0.6666667,0.6684917,6.011,
elapsed3,batch_004,TEMPTED,success,0.5555556,0.5555556,0.5529101,5.383,
elapsed4,batch_005,TEMPTED,success,0.5555556,0.5555556,0.5539683,6.066,
elapsed5,batch_006,TEMPTED,success,0.4761905,0.4761905,0.4585245,5.659,
elapsed6,batch_007,TEMPTED,success,0.6031746,0.6031746,0.5927437,7.929,
elapsed7,batch_008,TEMPTED,success,0.5079365,0.5079365,0.4966499,5.878,
elapsed8,batch_009,TEMPTED,success,0.4603175,0.4603175,0.4529915,12.056,
